# Feedback phase count ?? ?? ??

`unified_feedback_v4.py`? ???? phase sequence? ???? ?? count rule? ???? ???? notebook???.

?? ?? ??:

1. **?? phase ?? count**: `smooth_phase(window=5)` ? `UP` segment? `min_up_len=3` ???? count
2. **?? feedback alignment count**: `DOWN -> UP`? ?? ???? complete rep? count
3. **READY gap bridge ? alignment count**: ??? ??? `ready`? ?? ??? `DOWN -> UP`?? ???? ? count
4. **down ?? ?? count**: `UP` ?? `DOWN`? ? ??? count
5. **???? ?? count**: `UP` ??? threshold? ???? ?? count

?? ??? `infer/squat_15.json`? ??? `phase`???. ??? ?? phase? ?? ?? ??? ?? optional video inference cell?? `RUN_VIDEO_MODEL=True`? ??? ???.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional, Sequence
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from phase_bridge import (
    PHASE_READY,
    PHASE_DOWN,
    PHASE_UP,
    PHASE_NAMES,
    bridge_ready_gaps,
    count_phases_like_training,
    debounce_phase,
    phase_name,
    segment_reps,
    smooth_phase_like_training,
)

REPO = Path.cwd()
PHASE_TO_ID = {name: idx for idx, name in enumerate(PHASE_NAMES)}
ID_TO_PHASE = {idx: name for idx, name in enumerate(PHASE_NAMES)}
PHASE_CMAP = ListedColormap(["#666666", "#1f77b4", "#ff7f0e"])

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## 1) ?? phase ??

?? ?? ??? `unified_feedback_v4.py`? ??? ? expert JSON? ?? ????.

- `infer/squat_15.json`: `squat_15.mp4`?? ?? ??? expert cache
- ?? ???? `JSON_PATH`? ??? ???.
- GT ??? ?? ??? `GT_COUNT`? ??? ??? grid ??? ??? ?????.

In [ ]:
JSON_PATH = REPO / "infer" / "squat_15.json"
GT_COUNT: Optional[int] = None  # ?: ?? 5?? 5. ??? None

# phase? ?? JSON?? ?? optional video inference cell? ?? phase? ?????.
print(JSON_PATH)

In [ ]:
def load_phase_from_feedback_json(path: Path | str) -> tuple[np.ndarray, dict]:
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if "phase" not in data:
        raise KeyError(f"JSON? phase? ????: {path}")
    phase = np.asarray(data["phase"], dtype=np.int16).reshape(-1)
    phase = np.where((0 <= phase) & (phase <= 2), phase, PHASE_READY).astype(np.int8)
    return phase, data

raw_phase, source_data = load_phase_from_feedback_json(JSON_PATH)
print("frames:", len(raw_phase))
print("source_video:", source_data.get("source_video"))
print("stored reps:", len(source_data.get("reps") or []), "ref:", source_data.get("ref_rep_index"))

## 2) Phase sequence ??/??? ??

In [ ]:
def rle(seq: Sequence[int] | np.ndarray) -> list[tuple[int, int, int]]:
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    if arr.size == 0:
        return []
    runs = []
    start = 0
    for i in range(1, len(arr) + 1):
        if i == len(arr) or arr[i] != arr[start]:
            runs.append((int(arr[start]), int(start), int(i)))
            start = i
    return runs


def phase_summary(seq: Sequence[int] | np.ndarray, max_runs: int = 30) -> str:
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    if arr.size == 0:
        return "empty"
    vals, counts = np.unique(arr, return_counts=True)
    counts_s = ", ".join(f"{phase_name(int(v))}={int(c)}" for v, c in zip(vals, counts))
    runs = [f"{phase_name(l)}:{s}-{e}({e-s})" for l, s, e in rle(arr)[:max_runs]]
    suffix = "" if len(rle(arr)) <= max_runs else f" ... +{len(rle(arr))-max_runs} runs"
    return f"counts[{counts_s}] runs[{', '.join(runs)}{suffix}]"


def runs_df(seq: Sequence[int] | np.ndarray) -> pd.DataFrame:
    rows = []
    for idx, (label, start, end) in enumerate(rle(seq)):
        rows.append({
            "run": idx,
            "phase_id": label,
            "phase": phase_name(label),
            "start": start,
            "end_exclusive": end,
            "end_inclusive": end - 1,
            "len": end - start,
        })
    return pd.DataFrame(rows)


def plot_phase(seq: Sequence[int] | np.ndarray, title: str = "phase", events: Optional[Sequence[int]] = None, max_frames: Optional[int] = None):
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    if max_frames is not None:
        arr = arr[:max_frames]
    fig_w = max(10, min(24, len(arr) / 40)) if len(arr) else 10
    fig, ax = plt.subplots(figsize=(fig_w, 1.6))
    ax.imshow(arr[np.newaxis, :], aspect="auto", cmap=PHASE_CMAP, vmin=0, vmax=2)
    ax.set_yticks([])
    ax.set_title(title)
    ax.set_xlabel("frame")
    if events:
        for e in events:
            if 0 <= int(e) < len(arr):
                ax.axvline(int(e), color="red", linewidth=1.2, alpha=0.85)
    legend_handles = [plt.Line2D([0], [0], color=PHASE_CMAP(i), lw=6, label=name) for i, name in enumerate(PHASE_NAMES)]
    ax.legend(handles=legend_handles, loc="upper right", bbox_to_anchor=(1.0, 1.9), ncol=3)
    plt.show()

print(phase_summary(raw_phase))
plot_phase(raw_phase, "raw/stored phase")
runs_df(raw_phase).head(20)

## 3) Count rule ??

`training_up_end`? `model/train_ablation.py`? `count_phases`? ?? ?????.

??: ?? ?? ??? ??? `UP`?? ??? ??? terminal UP segment? count?? ????. ? notebook??? ????? `training_up_end_terminal`? ?? ?????.

In [ ]:
def count_up_segments(
    seq: Sequence[int] | np.ndarray,
    min_up_len: int = 3,
    count_terminal: bool = False,
    count_on: str = "segment_end",  # "segment_end" | "min_len_reached"
) -> tuple[int, list[int]]:
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    count = 0
    events: list[int] = []
    in_up = False
    up_start = -1
    counted_current = False

    for t, p in enumerate(arr):
        if int(p) == PHASE_UP:
            if not in_up:
                in_up = True
                up_start = t
                counted_current = False
            if count_on == "min_len_reached" and not counted_current and (t - up_start + 1) >= int(min_up_len):
                count += 1
                events.append(t)
                counted_current = True
        else:
            if in_up:
                up_len = t - up_start
                if count_on == "segment_end" and up_len >= int(min_up_len):
                    count += 1
                    events.append(t)
                in_up = False
                counted_current = False

    if count_terminal and in_up:
        up_len = len(arr) - up_start
        if count_on == "segment_end" and up_len >= int(min_up_len):
            count += 1
            events.append(len(arr) - 1)
    return count, events


def count_down_before_up(
    seq: Sequence[int] | np.ndarray,
    min_up_len: int = 3,
    min_down_len: int = 1,
    count_terminal: bool = False,
) -> tuple[int, list[int]]:
    """UP segment? ??, ?? count ?? DOWN segment? ??? ? ??? count."""
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    runs = rle(arr)
    seen_down = False
    count = 0
    events: list[int] = []
    for label, start, end in runs:
        length = end - start
        if label == PHASE_DOWN and length >= int(min_down_len):
            seen_down = True
        elif label == PHASE_UP:
            is_terminal = end == len(arr)
            if seen_down and length >= int(min_up_len) and (count_terminal or not is_terminal):
                count += 1
                events.append(len(arr) - 1 if is_terminal else end)
                seen_down = False
        elif label == PHASE_READY:
            # ready? seen_down? ???. ??? reset?? ??? ??? seen_down=False? ??? ?.
            pass
    return count, events


def count_current_alignment(seq: Sequence[int] | np.ndarray) -> tuple[int, list[int]]:
    reps = segment_reps(seq)
    # ?? HUD? top_frame? ??? +1?? ???.
    events = [int(rep.top_frame) for rep in reps if rep.top_frame is not None]
    return len(reps), events

## 4) ??? config + ?? count method ? ?? ??

??? ??:

1. optional smoothing: none / centered / causal
2. optional debounce: ?? phase run? ?? phase? ??
3. optional READY bridge: `DOWN READY UP` ?? `X READY X` ??

?? `unified_feedback_v4.py` alignment ?? ??:

```python
bridge_ready_gaps(debounce_phase(raw_phase, 4), 30)
segment_reps(...)
```

In [ ]:
def causal_majority(seq: Sequence[int] | np.ndarray, window: int = 5) -> np.ndarray:
    arr = np.asarray(seq, dtype=np.int16).reshape(-1)
    if int(window) <= 1 or len(arr) == 0:
        return arr.astype(np.int8).copy()
    out = np.zeros_like(arr, dtype=np.int8)
    for i in range(len(arr)):
        w = arr[max(0, i - int(window) + 1): i + 1]
        vals, counts = np.unique(w, return_counts=True)
        # tie? ?? phase ??
        max_count = counts.max()
        candidates = set(vals[counts == max_count].tolist())
        for x in w[::-1]:
            if int(x) in candidates:
                out[i] = int(x)
                break
    return out.astype(np.int8)


@dataclass(frozen=True)
class PrepConfig:
    smooth_mode: str = "none"   # "none" | "centered" | "causal"
    smooth_window: int = 5
    debounce_min_len: int = 1    # 1?? off
    ready_bridge_gap: int = 0    # 0?? off


@dataclass(frozen=True)
class CountConfig:
    min_up_len: int = 3
    min_down_len: int = 1


def prepare_phase(seq: Sequence[int] | np.ndarray, cfg: PrepConfig) -> np.ndarray:
    out = np.asarray(seq, dtype=np.int16).reshape(-1).astype(np.int8)
    if cfg.smooth_mode == "centered":
        out = smooth_phase_like_training(out, cfg.smooth_window)
    elif cfg.smooth_mode == "causal":
        out = causal_majority(out, cfg.smooth_window)
    elif cfg.smooth_mode == "none":
        out = out.copy()
    else:
        raise ValueError(f"unknown smooth_mode: {cfg.smooth_mode}")

    if int(cfg.debounce_min_len) > 1:
        out = debounce_phase(out, cfg.debounce_min_len)
    if int(cfg.ready_bridge_gap) > 0:
        out = bridge_ready_gaps(out, cfg.ready_bridge_gap)
    return out.astype(np.int8)


def evaluate_methods(seq: Sequence[int] | np.ndarray, prep: PrepConfig, count_cfg: CountConfig) -> tuple[pd.DataFrame, dict[str, np.ndarray], dict[str, list[int]]]:
    prepared = prepare_phase(seq, prep)
    rows = []
    sequences = {"prepared": prepared}
    events_by_method: dict[str, list[int]] = {}

    methods = []

    c, ev = count_phases_like_training(prepared, min_up_len=count_cfg.min_up_len)
    methods.append(("training_up_end_original", c, ev, "?? train_ablation count_phases: terminal UP ????"))

    c, ev = count_up_segments(prepared, min_up_len=count_cfg.min_up_len, count_terminal=True, count_on="segment_end")
    methods.append(("training_up_end_terminal", c, ev, "UP?? ??? ??? ??? UP count"))

    c, ev = count_up_segments(prepared, min_up_len=count_cfg.min_up_len, count_terminal=False, count_on="min_len_reached")
    methods.append(("realtime_up_min_reached", c, ev, "UP ??? min? ???? ?? count"))

    c, ev = count_down_before_up(prepared, min_up_len=count_cfg.min_up_len, min_down_len=count_cfg.min_down_len)
    methods.append(("down_before_up_nonadjacent", c, ev, "UP ?? DOWN? ? ??? count; READY? ??"))

    c, ev = count_current_alignment(prepared)
    methods.append(("current_down_up_adjacent", c, ev, "?? alignment?: DOWN ?? ?? UP complete rep"))

    for name, c, ev, note in methods:
        rows.append({
            "method": name,
            "count": int(c),
            "events": list(map(int, ev)),
            "note": note,
            "gt": GT_COUNT,
            "abs_error": None if GT_COUNT is None else abs(int(c) - int(GT_COUNT)),
        })
        events_by_method[name] = list(map(int, ev))
    return pd.DataFrame(rows), sequences, events_by_method

# ?? unified_feedback_v4.py alignment? ??? baseline
baseline_prep = PrepConfig(smooth_mode="none", smooth_window=5, debounce_min_len=4, ready_bridge_gap=30)
baseline_count = CountConfig(min_up_len=3, min_down_len=1)

baseline_df, baseline_sequences, baseline_events = evaluate_methods(raw_phase, baseline_prep, baseline_count)
display(baseline_df)
plot_phase(baseline_sequences["prepared"], "baseline prepared = debounce(4) + ready_bridge(30)", baseline_events["current_down_up_adjacent"])
print(phase_summary(baseline_sequences["prepared"]))

## 5) Grid search

?? grid?? smoothing/window/gap/min length? ???? count ??? ? ?? ???.

`GT_COUNT`? ??? ??? `abs_error` ???? ?????.

In [ ]:
SMOOTH_MODES = ["none", "centered", "causal"]
SMOOTH_WINDOWS = [1, 3, 5, 7]
DEBOUNCE_MIN_LENS = [1, 3, 4, 5]
READY_BRIDGE_GAPS = [0, 10, 20, 30, 45]
MIN_UP_LENS = [2, 3, 4, 5]
MIN_DOWN_LENS = [1, 3, 5]

all_rows = []
for smooth_mode in SMOOTH_MODES:
    for smooth_window in SMOOTH_WINDOWS:
        if smooth_mode == "none" and smooth_window != 1:
            continue
        for debounce_min_len in DEBOUNCE_MIN_LENS:
            for ready_bridge_gap in READY_BRIDGE_GAPS:
                prep = PrepConfig(
                    smooth_mode=smooth_mode,
                    smooth_window=smooth_window,
                    debounce_min_len=debounce_min_len,
                    ready_bridge_gap=ready_bridge_gap,
                )
                for min_up_len in MIN_UP_LENS:
                    for min_down_len in MIN_DOWN_LENS:
                        cfg = CountConfig(min_up_len=min_up_len, min_down_len=min_down_len)
                        df, _, _ = evaluate_methods(raw_phase, prep, cfg)
                        for row in df.to_dict("records"):
                            row.update({
                                "smooth_mode": smooth_mode,
                                "smooth_window": smooth_window,
                                "debounce_min_len": debounce_min_len,
                                "ready_bridge_gap": ready_bridge_gap,
                                "min_up_len": min_up_len,
                                "min_down_len": min_down_len,
                            })
                            all_rows.append(row)

grid_df = pd.DataFrame(all_rows)
sort_cols = ["abs_error", "method", "smooth_mode", "smooth_window"] if GT_COUNT is not None else ["method", "count", "smooth_mode", "smooth_window"]
display(grid_df.sort_values(sort_cols).head(80))
print("rows:", len(grid_df))

## 6) ??? count? ????? ??

?: count? 5? ??? ??, ?? method? ??.

In [ ]:
TARGET_COUNT: Optional[int] = None  # ?: 5
METHOD_FILTER: Optional[str] = None  # ?: "current_down_up_adjacent" ?? "training_up_end_original"

view = grid_df.copy()
if TARGET_COUNT is not None:
    view = view[view["count"] == int(TARGET_COUNT)]
if METHOD_FILTER:
    view = view[view["method"] == METHOD_FILTER]

cols = [
    "method", "count", "abs_error", "smooth_mode", "smooth_window", "debounce_min_len", "ready_bridge_gap",
    "min_up_len", "min_down_len", "events", "note",
]
display(view[cols].sort_values(["method", "smooth_mode", "smooth_window", "debounce_min_len", "ready_bridge_gap"]).head(200))

## 7) ??? config ???

?? ?? ??? phase timeline? count ??? ??? ?????.

In [ ]:
SELECTED_PREP = PrepConfig(
    smooth_mode="none",
    smooth_window=5,
    debounce_min_len=4,
    ready_bridge_gap=30,
)
SELECTED_COUNT = CountConfig(min_up_len=3, min_down_len=1)
SELECTED_METHOD = "current_down_up_adjacent"

selected_df, selected_sequences, selected_events = evaluate_methods(raw_phase, SELECTED_PREP, SELECTED_COUNT)
display(selected_df)
prepared = selected_sequences["prepared"]
plot_phase(raw_phase, "raw phase")
plot_phase(prepared, f"prepared phase | {SELECTED_PREP}", selected_events.get(SELECTED_METHOD))
display(runs_df(prepared).head(80))

## 8) ??? synthetic sequence? rule ?? ??

?? phase list? ???? count rule ??? ??? ? ????.

In [ ]:
# 0=ready, 1=down, 2=up
synthetic = np.array([
    0,0,
    1,1,1,
    0,0,
    2,2,2,2,
    0,0,
    1,1,1,
    2,2,2,
    0,
], dtype=np.int8)

syn_prep = PrepConfig(smooth_mode="none", debounce_min_len=1, ready_bridge_gap=0)
syn_df, syn_seq, syn_events = evaluate_methods(synthetic, syn_prep, CountConfig(min_up_len=3, min_down_len=1))
display(syn_df)
plot_phase(synthetic, "synthetic raw")

syn_bridge_prep = PrepConfig(smooth_mode="none", debounce_min_len=1, ready_bridge_gap=3)
syn_bridge_df, syn_bridge_seq, syn_bridge_events = evaluate_methods(synthetic, syn_bridge_prep, CountConfig(min_up_len=3, min_down_len=1))
display(syn_bridge_df)
plot_phase(syn_bridge_seq["prepared"], "synthetic after ready_bridge_gap=3", syn_bridge_events["current_down_up_adjacent"])

## 9) Optional: ??? ???? phase ?? ??

?? cell? ?????. `RUN_VIDEO_MODEL=True`? ??? `unified_feedback_v4.py`? MediaPipe ?? ??? `PhaseModelAdapter`? ???? ???? phase? ?? ????.

??:

- conda env `capstone`?? `mediapipe`, `torch`, `opencv`? ??? ???? ?????.
- ?? ? `raw_phase = video_phase`? ??? ? grid? ??? ?? ???? ?? ?? ? ????.

In [ ]:
RUN_VIDEO_MODEL = False
VIDEO_PATH = REPO / "infer" / "squat_user1.mp4"
EXERCISE = "squat"

if RUN_VIDEO_MODEL:
    from unified_feedback_v4 import (
        MODEL_CKPT,
        MODEL_DEVICE,
        MODEL_CLIP_LEN,
        MODEL_STRIDE,
        MODEL_INPUT_KIND,
        MODEL_GRAPH,
        WINDOW_ANCHOR,
        MODEL_PATH,
        LANDMARK_EMA_ALPHA,
        VIS_THR,
        open_video_normalized,
        apply_rotation,
        create_landmarker,
        extract_all_landmarks,
        PersonSelector,
        LandmarkSmoother,
        SideLock,
        valid_lms,
    )
    from phase_bridge import PhaseModelAdapter, pose_seq_from_landmarks

    adapter = PhaseModelAdapter(
        ckpt_path=MODEL_CKPT,
        device=MODEL_DEVICE,
        clip_len=MODEL_CLIP_LEN,
        stride=MODEL_STRIDE,
        input_kind=MODEL_INPUT_KIND,
        graph=MODEL_GRAPH,
        anchor=WINDOW_ANCHOR,
        required=True,
    )
    cap, rot = open_video_normalized(str(VIDEO_PATH))
    if not cap.isOpened():
        raise RuntimeError(f"failed to open video: {VIDEO_PATH}")
    fps = cap.get(5) or 30.0
    total = int(cap.get(7) or 0)
    selector = PersonSelector(EXERCISE)
    smoother = LandmarkSmoother(LANDMARK_EMA_ALPHA, VIS_THR)
    side_lock = SideLock(EXERCISE)
    lms_seq = []
    with create_landmarker(MODEL_PATH) as landmarker:
        idx = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = apply_rotation(frame, rot)
            timestamp_ms = int(idx * 1000 / fps)
            raw = selector.select(extract_all_landmarks(landmarker, frame, timestamp_ms))
            lms = smoother.update(raw)
            _side = side_lock.update(lms)
            lms_seq.append(lms if valid_lms(lms) else None)
            idx += 1
            if idx % 50 == 0:
                print(f"scan {idx}/{total if total else '?'}")
    cap.release()

    pose_seq = pose_seq_from_landmarks(lms_seq)
    result = adapter.infer(pose_seq)
    video_phase = result.phase_per_frame.astype(np.int8)
    print("action:", result.action, "conf:", result.action_conf, "frames:", len(video_phase))
    print(phase_summary(video_phase))
    plot_phase(video_phase, f"video raw phase: {VIDEO_PATH.name}")

    # ?? ? cell?? ??? ?? ???? ?? ??? ??? ?? ??
    # raw_phase = video_phase

## 10) ?? integration? ?? ?? count? ? ?? ??

`training_count`? ?? ?? `UP` segment? ?? ? ??? ??? ????.  
`align_count`? ?? ??? `DOWN -> UP`? `READY` ??? ????, DOWN/UP ? ??? ?? ????.

In [ ]:
training_phase = smooth_phase_like_training(raw_phase, window=5)
training_count, training_events = count_phases_like_training(training_phase, min_up_len=3)

align_phase = bridge_ready_gaps(debounce_phase(raw_phase, 4), 30)
align_reps = segment_reps(align_phase)
align_count = len(align_reps)
align_events = [rep.top_frame for rep in align_reps if rep.top_frame is not None]

print("training_count:", training_count, "events:", training_events[:30])
print("align_count:", align_count, "events:", align_events[:30])
plot_phase(training_phase, "training phase = centered smooth window 5", training_events)
plot_phase(align_phase, "alignment phase = debounce 4 + ready bridge 30", align_events)